In [ ]:
'''
This notebook sets up the resampling of Sentinel-2 L1B data from 
the instrument grid (pixels on the detector field of view) into 
regularly spaced image grid data, as a first step to producing 
orthorectified data.

This script was originally intended for internal use and may have
limited functionality, bugs and other problems. It does not have 
detailed documentation. Please contact the ESA Sentinel-2 Next Generation 
mission scientist for more information: simon{dot}proud{at}esa{dot}int

Please also note, some directory structures in this notebook are
hardcoded and will require modification. Check the whole notebook
carefully before running on your system.

IMPORTANT NOTE: This script generates a runner_gdal file that will actually
run the resampling via gdal on the command line. The script itself *does not* 
do any processing. Ideally, this would be fixed in a future version, to call
gdal directly via the python bindings.

Once the script is complete, run the `runner_gdal` file from your terminal.
'''

from pathlib import Path
from glob import glob
import os

# Input directory, containing Sentinel-2 L1B files *AFTER* processing with Sen2VM
# This script will fail if the files generated by Sen2VM are not present!
idir = "/data/"

# Output directory. Currently not used, output is stored in the top level directory
# of each Sentinel-2 L1B granule.
odir = "/data/output/"

In [ ]:
# Find S2 L1b granules
indirs = glob(f"{idir}/DATA/S2*/")

indirs = [d for d in indirs if os.path.basename(os.path.normpath(d)) != "bk"]
indirs.sort()

print(f"Found a total of {len(indirs)} input scenes.")

Found a total of 56 input scenes.


In [ ]:
# We use gdalwarp to resample the data
base_gdal = "gdalwarp SENTINEL2_L1B_WITH_GEOLOC"

# Note: This uses weighted averaging resampling. Change the `-r average`
# to a different method as needed. The other parameters control parallelisation
# and memory use. They are optimised for the system used to process the night 
# data and may require adaptation on your own machine.
end_gdal = " -r average -multi -wm 5000M -wo NUM_THREADS=8 -dstnodata 65535 "

counter = 0

# Select which bands to process
good_bands = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B8A", "B08", "B09", "B10", "B11", "B12"]
#good_bands = ["B01", "B02"]

print(f"Beginning processing of {len(indirs)} scenes")

overall_counter = 0

# Run the main processing loop
with open("/{idir}/runner_gdal", "w") as fid:
    for cdir in indirs:
        main_counter = 0
        done_counter = 0
        Path(f"{odir}/{cdir[12:14]}/").mkdir(parents=True, exist_ok=True)
        ds_dir = glob(f"{cdir}/DATASTRIP/S*")
        ds_dir.sort()
        if len(ds_dir) < 1:
            print(f"Can't find datastrip dir: {cdir}")
            continue
        else:
            ds_dir = ds_dir[0]
        ds_mtd = glob(f"{cdir}/S2A*.xml")
        if len(ds_mtd) < 1:
            print(f"Can't find datastrip metadata: {cdir}")
            continue
        else:
            ds_mtd = ds_mtd[0]
        if ds_dir[-4:] == ".tar":
            ds_dir = ds_dir[:-4]
        geo_files = glob(f"{ds_dir}/GEO_DATA/*.vrt")
        if len(geo_files) <  1:
            print(f" - Not enough GEO files for {cdir[len(idir):-6]}")
            continue
        for geo_file in geo_files:
            pos = geo_file.find("S2A_OPER_GEO")
            if geo_file[-7:-4] not in good_bands:
                continue
            filestr = geo_file[pos:-4]
            outf = f"{cdir}/{filestr}.tif"
            if os.path.exists(outf):
                done_counter += 1
                continue
            mainstr = f"{base_gdal}:{ds_mtd}:{filestr} {outf} {end_gdal}"
            if counter < 5:
                mainstr = f"{mainstr} &\n"
                counter += 1
            else:
                counter = 0
                mainstr = f"{mainstr}\n"
            fid.write(mainstr)
            main_counter += 1
        if main_counter > 0:
            overall_counter += main_counter
            print(f" - Discovered {main_counter} unprocessed band/det combinations, adding to queue.")
        elif done_counter == len(geo_files):
            print(f" - Already processed all band/det combinations for this scene.")

print(f"Ending processing, successful. There are {overall_counter} files to process.")

Beginning processing of 56 scenes
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed all band/det combinations for this scene.
 - Already processed 

12:48 6720
12:52 6552
13:00 6275
13:09 5934
14:11 3985
14:49 2880
16:56 2340
17:05 1896
17:20 1606
18:14 1274
18:29 0772
18:41 0503
19:09 0001